# Lift evaluation v1: actor-initialized action-space CEM

Uses the same trained actor/world-model checkpoints as run_primary.ipynb; no retraining.
At each real replan the actor generates one complete goal-conditioned plan.
Ordinary CEM starts with this plan as its mean and refits Gaussian action distributions.
The actor is not called during CEM candidate refinement. No residual temperature.
Goal stages, execution of five primitives, fixed allowances, and videos are unchanged.


In [ ]:
%pip install --upgrade pip setuptools wheel
%pip install h5py imageio imageio-ffmpeg "mujoco==3.2.6" "robosuite==1.4.1"
%pip install --no-deps "git+https://github.com/NVlabs/mimicgen.git@72bd767c255545f462e7ccfb2731f2e5d4c1d9bb"

# Fail in the setup cell with a clear message instead of much later.
import mujoco
import robosuite
import mimicgen
print(f'MuJoCo {mujoco.__version__} installed successfully')
print(f'robosuite {robosuite.__version__} installed successfully')
print('MimicGen installed successfully')


## Google Drive / local eval setup

Put your existing evals.hdf5 and subgoal_labels.json in MyDrive/primary_actor.
Checkpoints load from primary_actor/models; videos save to primary_actor/rollouts.
No training dataset download is needed. EVAL_HDF5 and SUBGOAL_LABELS can instead point
to files uploaded directly into Colab local storage.
Choose a GPU runtime. The final rollout cell reloads all networks and goal embeddings
when selecting a different checkpoint.


In [ ]:
import ctypes
import json
import re
import tempfile
import time
import zipfile
from collections import deque
from pathlib import Path

import h5py
import imageio.v2 as imageio
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from google.colab import drive

drive.mount('/content/drive')
PRIMARY_DIR = Path('/content/drive/MyDrive/primary_actor')
RUN_DIR = PRIMARY_DIR
MODEL_DIR = RUN_DIR / 'models'
ROLLOUT_DIR = RUN_DIR / 'rollouts'
ROLLOUT_DIR.mkdir(parents=True, exist_ok=True)

WINDOW = 3
EMBEDDING_DIM = 192
FRAME_SKIP = 5
PRIMITIVE_ACTION_DIM = 7
PREDICTOR_ACTION_DIM = FRAME_SKIP * PRIMITIVE_ACTION_DIM
CONTROL_FREQ = 20
ENV_NAME = 'Lift_D0'
CHECKPOINT_FORMAT_VERSION = 5
ARCHITECTURE_ID = 'lewm_robot_goal_actor_action64_v5'
ACTION_SMOOTHING_DIM = 64
ACTION_ENCODING_ID = 'standardized_ordered_primitive_block_v2'
IMAGE_NORMALIZATION_ID = 'imagenet'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {DEVICE}')

def normalize_robot_images(images):
    shape = [1] * images.ndim
    shape[-3] = 3
    mean = images.new_tensor((0.485, 0.456, 0.406)).reshape(shape)
    std = images.new_tensor((0.229, 0.224, 0.225)).reshape(shape)
    return (images - mean) / std

def build_robot_state(pos, quat, grip):
    quat = quat / quat.norm(dim=-1, keepdim=True).clamp_min(1e-8)
    return torch.cat((pos, quat, grip), dim=-1)

def normalize_robot_state(state, mean, std):
    return (state - mean) / std



In [ ]:
import torch
import torch.nn.functional as F
from torch import nn


class ProjectionMLP(nn.Module):
    """LeWM projection head used after encoder and predictor features."""

    def __init__(self, input_dim: int = 192, hidden_dim: int = 2048, output_dim: int = 192):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, x):
        return self.layers(x)


class ActionEncoder(nn.Module):
    """Encode one ordered primitive-action block for AdaLN conditioning."""

    def __init__(self, input_dim: int, smoothed_dim: int = 10, embedding_dim: int = 192):
        super().__init__()
        self.smoother = nn.Conv1d(input_dim, smoothed_dim, kernel_size=1)
        self.mlp = nn.Sequential(
            nn.Linear(smoothed_dim, 4 * embedding_dim),
            nn.SiLU(),
            nn.Linear(4 * embedding_dim, embedding_dim),
        )

    def forward(self, actions):
        batch_size, sequence_length, _ = actions.shape
        actions = self.smoother(actions.transpose(1, 2)).transpose(1, 2)
        return self.mlp(actions.reshape(batch_size * sequence_length, -1)).reshape(
            batch_size, sequence_length, -1
        )


class TransformerBlock(nn.Module):
    """Pre-norm transformer block with optional AdaLN-Zero conditioning."""

    def __init__(
        self,
        d_model: int = 192,
        nheads: int = 16,
        d_action: int | None = None,
        dropout: float = 0.1,
        causal_masking: bool = True,
        adaptive: bool = True,
        dim_head: int | None = None,
        mlp_dim: int | None = None,
    ):
        super().__init__()
        self.adaptive = adaptive
        hidden_dim = mlp_dim or 4 * d_model

        self.norm_one = nn.LayerNorm(d_model, elementwise_affine=not adaptive)
        self.norm_two = nn.LayerNorm(d_model, elementwise_affine=not adaptive)
        if adaptive:
            if d_action is None:
                raise ValueError("d_action is required for an adaptive transformer block")
            self.adaLN_modulation = nn.Sequential(
                nn.SiLU(),
                nn.Linear(d_action, 6 * d_model),
            )
            nn.init.zeros_(self.adaLN_modulation[-1].weight)
            nn.init.zeros_(self.adaLN_modulation[-1].bias)

        self.attn_block = MultiHeadAttention(
            d_model=d_model,
            nheads=nheads,
            dim_head=dim_head,
            dropout=dropout,
            causal_masking=causal_masking,
        )
        self.mlp = nn.Sequential(
            nn.Linear(d_model, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, d_model),
            nn.Dropout(dropout),
        )

    @staticmethod
    def _modulate(x, shift, scale):
        return x * (1 + scale) + shift

    def forward(self, z, a=None):
        if self.adaptive:
            if a is None:
                raise ValueError("action conditioning is required for an adaptive block")
            shift_attn, scale_attn, gate_attn, shift_mlp, scale_mlp, gate_mlp = (
                self.adaLN_modulation(a).chunk(6, dim=-1)
            )
            z = z + gate_attn * self.attn_block(
                self._modulate(self.norm_one(z), shift_attn, scale_attn)
            )
            z = z + gate_mlp * self.mlp(
                self._modulate(self.norm_two(z), shift_mlp, scale_mlp)
            )
        else:
            z = z + self.attn_block(self.norm_one(z))
            z = z + self.mlp(self.norm_two(z))
        return z


class MultiHeadAttention(nn.Module):
    def __init__(
        self,
        d_model: int = 192,
        nheads: int = 16,
        dim_head: int | None = None,
        dropout: float = 0.1,
        causal_masking: bool = True,
    ):
        super().__init__()
        self.dropout = dropout
        self.d_k = dim_head or (d_model // nheads)
        self.nheads = nheads
        self.inner_dim = self.nheads * self.d_k
        self.causal_masking = causal_masking

        self.W_q = nn.Linear(d_model, self.inner_dim)
        self.W_k = nn.Linear(d_model, self.inner_dim)
        self.W_v = nn.Linear(d_model, self.inner_dim)
        self.W_o = nn.Linear(self.inner_dim, d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        batch_size, sequence_length, _ = x.shape
        queries = self.W_q(x).reshape(
            batch_size, sequence_length, self.nheads, self.d_k
        ).transpose(1, 2)
        keys = self.W_k(x).reshape(
            batch_size, sequence_length, self.nheads, self.d_k
        ).transpose(1, 2)
        values = self.W_v(x).reshape(
            batch_size, sequence_length, self.nheads, self.d_k
        ).transpose(1, 2)

        attention = F.scaled_dot_product_attention(
            queries,
            keys,
            values,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=self.causal_masking,
        )
        attention = attention.transpose(1, 2).reshape(
            batch_size, sequence_length, self.inner_dim
        )
        return self.drop(self.W_o(attention))

from torch import nn
import torch

"""
Thought it would be fun to define the architecture of the ViT-Tiny
from scratch so here it is using the transformers I defined as well
"""

class CustomVitTiny(nn.Module):
    def __init__(self,
                 patch : int = 14,
                 nheads : int = 3,
                 nlayers : int = 12,
                 d_model : int = 192,
                 npatches : int = 256):
        super().__init__()

        self.npatches = npatches
        self.d_model = d_model

        self.conv_flattener = nn.Conv2d(in_channels=3,
                                        out_channels=d_model,
                                        kernel_size=patch,
                                        stride=patch,
                                        padding=0)
        
        self.transformer_layers = nn.ModuleList([])
        for _ in range(nlayers):
            self.transformer_layers.append(TransformerBlock(d_model=d_model, 
                                           nheads=nheads,                                            
                                           dropout=0.0,
                                           causal_masking=False,
                                           adaptive=False))
        # learned positional embeddings
        self.position_emb = nn.Parameter(torch.empty(npatches + 1, d_model))
        # learned [cls] embedding
        self.cls_emb = nn.Parameter(torch.empty(1, 1, 1, d_model))
        self.norm = nn.LayerNorm(d_model)
        nn.init.trunc_normal_(self.position_emb, std=0.02)
        nn.init.trunc_normal_(self.cls_emb, std=0.02)

    def forward(self, x):
        # shape of x: (B, N, C, H, W)
        B, N, C, H, W = x.shape
        x = x.reshape(B*N, C, H, W)
        emb = self.conv_flattener(x)
        emb = emb.reshape(B, N, self.d_model, self.npatches).permute(0, 1, 3, 2)
        # shape of emb now: (512, 3, 256, 192)
        cls_emb = self.cls_emb.expand(B, N, 1, self.d_model)
        emb = torch.cat([cls_emb, emb], dim=2)
        emb = emb + self.position_emb
        # shape of emb now: (512, 3, 257, 192)
        emb = emb.reshape(B*N, self.npatches + 1, self.d_model) # now ready to go into the transformer

        for layer in (self.transformer_layers):
           emb = layer(emb)
        emb = self.norm(emb)

        emb = emb.reshape(B, N, self.npatches + 1, self.d_model)
        emb = emb[:, :, 0, :] # shape : (512, 3, 192), this is what the predictor expects as the input

        return emb





# TEST CODE for Shape
# dummy_conv = nn.Conv2d(in_channels=3,
#           out_channels=192,
#           kernel_size=14,
#           stride=14,
#           padding=0)

# dummy_frame = torch.ones((512, 3, 3, 224, 224))
# dummy_frame = dummy_frame.reshape(512*3, 3, 224, 224)
# emb = dummy_conv(dummy_frame)

# print(emb.shape)
# print("after reshaping")
# emb = emb.reshape((512, 3, 192, 256))
# emb = emb.permute(0, 1, 3, 2)
# print(emb.shape)
# cls = torch.ones(1, 1, 1, 192)
# cls = cls.expand(512, 3, 1, cls.shape[-1])
# emb = torch.cat([cls, emb], dim=2)
# print(emb.shape)
# emb = emb[:, :, 0, :]
# print(emb.shape)

# TEST CODE for the Architecture
# test = CustomVitTiny()
# dummy = torch.ones(2, 3, 3, 224, 224)
# dummy = test(dummy)
# print(dummy.shape)

from torch import nn
import torch

"""
This is the network definition for the encoder network which in the paper
was just ViT-Tiny which is a visual transformer with the following parameters:
- patch size of 14, 12 layers, 3 attention heads, and a hidden dimension of 192
- there is a "projection" layer added to the end of this because visual transformers
  typically have layernorms at the end but this ruins the regularization used to
  prevent representation collapse
    - this projection layer is a single layer with batch normalization to undo the
      affects of the layernorm at the end
"""
# right now the image size is defaulted to 224x224

class EncoderNetwork(nn.Module):
    def __init__(self,
                 patch : int = 14,
                 nheads : int = 3,
                 nlayers : int = 12,
                 d_model : int = 192,
                 npatches : int = 256,
                 projector_hidden_dim: int = 2048,
                 project_output: bool = True): # npatches is (H/patch_size)*(W/patch_size)
        super().__init__()
        self.d_model = d_model
        
        self.vit = CustomVitTiny(patch, nheads, nlayers, d_model, npatches)
        self.projector = (
            ProjectionMLP(d_model, projector_hidden_dim, d_model)
            if project_output else nn.Identity()
        )
    
    def forward(self, x):
        # shape of input (x): (B, N, C, H, W)
        B, N, C, H, W = x.shape
        z = self.vit(x)
        #projection step
        z = z.reshape(B*N, self.d_model)
        z = self.projector(z)
        z = z.reshape(B, N, self.d_model)

        return z
    

# TEST Code
# dummy_image = torch.ones(24, 3, 3, 224, 224)
# enc = EncoderNetwork()
# print(enc(dummy_image).shape)

from torch import nn
import torch




"""
This is the modified encoder network that extends it to take in
2 camera observations plus robot state information and then output
a final embedding in latent space. Also modifying it to be 84x84 to match
the mimicgen dataset instead of 224x224 which was for the original definition.
This encoder will now seperately pass through each input image through
the default encoder and then concatenate them together with the robot state
embedding and then this concatenated vector will be passed through a fusion MLP
to compress it back to the original embedding space. The output of this encoder
will still be the standard dimension so it can be passed through the default
predictor
"""


class RobotEncoder(nn.Module):
    def __init__(self,
                 patch : int = 14,
                 nheads : int = 3,
                 nlayers : int = 12,
                 d_model : int = 192,
                 d_state : int = 9,
                 d_state_emb : int = 64,
                 npatches : int = 36,
                 projector_hidden_dim: int = 2048):  # 84x84, patch 14 → 6x6 patches
        super().__init__()
        encoder_kwargs = dict(
            patch=patch,
            nheads=nheads,
            nlayers=nlayers,
            d_model=d_model,
            npatches=npatches,
            projector_hidden_dim=projector_hidden_dim,
            project_output=False,
        )
        # The architectures are identical, but no parameters are shared. This
        # lets each visual backbone specialize to its camera's representation.
        self.agent_encoder = EncoderNetwork(**encoder_kwargs)
        self.wrist_encoder = EncoderNetwork(**encoder_kwargs)

        # small projection network for the 
        self.state_encoder = nn.Sequential(
            nn.Linear(in_features=d_state, out_features=2*d_state),
            nn.GELU(),
            nn.Linear(in_features=2*d_state, out_features=d_state_emb)
        )

        self.fusion_mlp = nn.Sequential(
            nn.Linear(in_features=(d_model * 2 + d_state_emb), out_features=(d_model * 2)),
            nn.GELU(),
            nn.Linear(in_features=(d_model * 2), out_features=d_model),
        )
        self.projector = ProjectionMLP(d_model, projector_hidden_dim, d_model)

    def forward(self, above, wrist, state):
        # above : [Batch, Window, 3, 84, 84] | wrist : [Batch, Window, 3, 84, 84] | state : [Batch, Window, 9]
        above_emb = self.agent_encoder(above)
        wrist_emb = self.wrist_encoder(wrist)
        state_emb = self.state_encoder(state) # [Batch, Window, 64]
        fusion = torch.cat((above_emb, wrist_emb, state_emb), dim=-1) # [Batch, Window, 448]
        fusion = self.fusion_mlp(fusion)
        flat = fusion.reshape(-1, fusion.shape[-1])
        return self.projector(flat).reshape_as(fusion) # [Batch, Window, 192]

import torch
from torch import nn





class PredictorNetwork(nn.Module):
    """Action-conditioned latent predictor matching the released LeWM design."""

    def __init__(
        self,
        d_model: int = 192,
        nheads: int = 16,
        d_action: int = 2,
        dropout: float = 0.1,
        nlayers: int = 6,
        window: int = 3,
        dim_head: int = 64,
        mlp_dim: int = 2048,
        action_smoothing_dim: int = 10,
        projector_hidden_dim: int = 2048,
    ):
        super().__init__()
        self.d_model = d_model
        self.d_action = d_action
        self.window = window
        self.action_encoder = ActionEncoder(
            input_dim=d_action,
            smoothed_dim=action_smoothing_dim,
            embedding_dim=d_model,
        )
        self.transformer_layers = nn.ModuleList(
            [
                TransformerBlock(
                    d_model=d_model,
                    nheads=nheads,
                    d_action=d_model,
                    dropout=dropout,
                    causal_masking=True,
                    adaptive=True,
                    dim_head=dim_head,
                    mlp_dim=mlp_dim,
                )
                for _ in range(nlayers)
            ]
        )
        self.position_emb = nn.Parameter(torch.empty(1, window, d_model))
        nn.init.trunc_normal_(self.position_emb, std=0.02)
        self.final_norm = nn.LayerNorm(d_model)
        self.projector = ProjectionMLP(d_model, projector_hidden_dim, d_model)

    def forward(self, z, a):
        batch_size, sequence_length, embedding_dim = z.shape
        if embedding_dim != self.d_model:
            raise ValueError(f"expected embedding size {self.d_model}, got {embedding_dim}")
        if sequence_length > self.window:
            raise ValueError(f"sequence length {sequence_length} exceeds window {self.window}")
        if a.shape[:2] != z.shape[:2] or a.shape[-1] != self.d_action:
            raise ValueError(
                f"expected actions [B, {sequence_length}, {self.d_action}], got {tuple(a.shape)}"
            )

        z = z + self.position_emb[:, :sequence_length]
        action_embedding = self.action_encoder(a)
        for layer in self.transformer_layers:
            z = layer(z, action_embedding)
        z = self.final_norm(z)
        z = self.projector(z.reshape(batch_size * sequence_length, self.d_model))
        return z.reshape(batch_size, sequence_length, self.d_model)


"""Goal-conditioned autoregressive policy over standardized primitive actions.

Fixed five-action adaptation, not a complete FlexiWorld reproduction. The actor
sees the current fused latent, goal intent, their product, the previous action
block, and the shifted within-block prefix. It never sees the future state.
"""
import math

import torch
from torch import nn


class GoalActor(nn.Module):
    def __init__(self, d_model=192, action_dim=7, block_size=5,
                 nlayers=3, nheads=4, ff_dim=768, dropout=0.1):
        super().__init__()
        self.d_model, self.action_dim, self.block_size = d_model, action_dim, block_size
        self.config = dict(d_model=d_model, action_dim=action_dim, block_size=block_size,
                           nlayers=nlayers, nheads=nheads, ff_dim=ff_dim, dropout=dropout)
        self.context = nn.ModuleList([nn.Linear(d_model, d_model) for _ in range(3)])
        self.previous = nn.Sequential(nn.Linear(block_size * action_dim, 64), nn.GELU(),
                                      nn.Linear(64, d_model))
        self.action = nn.Linear(action_dim, d_model)
        self.bos = nn.Parameter(torch.zeros(1, 1, d_model))
        self.position = nn.Parameter(torch.randn(1, 4 + block_size, d_model) * 0.02)
        layer = nn.TransformerEncoderLayer(d_model, nheads, ff_dim, dropout,
                                           activation='gelu', batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(layer, nlayers, norm=nn.LayerNorm(d_model),
                                                 enable_nested_tensor=False)
        self.mean = nn.Linear(d_model, action_dim)
        self.log_std = nn.Linear(d_model, action_dim)

    def forward(self, current, goal, previous, prefix):
        """prefix [B,L,7] predicts L+1 actions (max five), using only past tokens."""
        if prefix.shape[1] >= self.block_size:
            raise ValueError('Prefix must be shorter than the action block')
        intent = goal - current
        context = torch.stack([p(x) for p, x in zip(
            self.context, (current, intent, current * intent))] +
            [self.previous(previous.flatten(1))], dim=1)
        tokens = torch.cat((context, self.bos.expand(current.shape[0], -1, -1),
                            self.action(prefix)), dim=1)
        tokens = tokens + self.position[:, :tokens.shape[1]]
        causal = torch.ones(tokens.shape[1], tokens.shape[1], dtype=torch.bool,
                            device=tokens.device).triu(1)
        hidden = self.transformer(tokens, mask=causal)[:, 4:]
        return self.mean(hidden), self.log_std(hidden).clamp(-5, 2)

    @torch.no_grad()
    def generate(self, current, goal, previous):
        prefix = current.new_empty(current.shape[0], 0, self.action_dim)
        for _ in range(self.block_size):
            mean, _ = self(current, goal, previous, prefix)
            prefix = torch.cat((prefix, mean[:, -1:]), dim=1)
        return prefix


def actor_objective(actor, current, goal, previous, targets, valid, student_probability=0.0):
    """Masked per-sample Gaussian NLL; Student Forcing only changes prefixes.

    Goal branch detached; current branch remains attached. No privileged state.
    Validation should pass student_probability=0 for deterministic teacher forcing.
    """
    prefix = targets[:, :-1]
    if student_probability > 0:
        was_training = actor.training
        actor.eval()
        try:
            generated = actor.generate(current.detach(), goal.detach(), previous.detach())
        finally:
            actor.train(was_training)
        select = torch.rand(targets.shape[0], 1, 1, device=targets.device) < student_probability
        prefix = torch.where(select, generated[:, :-1], prefix)
    mean, log_std = actor(current, goal.detach(), previous, prefix)
    mean, log_std, targets = mean.float(), log_std.float(), targets.float()
    nll = 0.5 * ((targets - mean) * torch.exp(-log_std)).square() + log_std + 0.5 * math.log(2 * math.pi)
    mask = valid.float()
    denominator = mask.sum(1).clamp_min(1)
    loss = ((nll.mean(-1) * mask).sum(1) / denominator).mean()
    mse = (((targets - mean).square().mean(-1) * mask).sum(1) / denominator).mean()
    return loss, mse


In [ ]:
class Planner:
	def __init__(
		self,
		device,
		enc: nn.Module,
		pred: nn.Module,
		H: int = 5,
		N: int = 300,
		K: int = 30,
		T: int = 10,
		action_dim: int = 35,
		primitive_action_dim: int = 7,
		action_mean=None,
		action_std=None,
		action_low=None,
		action_high=None,
		raw_action_mean=None,
		raw_action_std=None,
		min_std=0.05,
		momentum: float = 0.1,
		seed: int | None = None,
		noise_correlation: float = 0.7,
		fresh_fraction: float = 0.1,
		separate_gripper_searches: bool = True,
	):
		self.device = torch.device(device)
		self.enc = enc.to(self.device)
		self.pred = pred.to(self.device)
		self.H, self.N, self.K, self.T = H, N, K, T
		self.action_dim = action_dim
		self.primitive_action_dim = primitive_action_dim
		if min(H, N, K, T, action_dim, primitive_action_dim) < 1:
			raise ValueError("planner dimensions and CEM settings must be positive")
		if K > N:
			raise ValueError(f"K ({K}) cannot exceed N ({N})")
		if action_dim % primitive_action_dim:
			raise ValueError("action_dim must contain complete primitive-action blocks")
		self.block_size = action_dim // primitive_action_dim
		self.gripper_indices = np.arange(
			primitive_action_dim - 1, action_dim, primitive_action_dim
		)

		# CEM works in the exact standardized coordinates consumed by the model.
		self.action_mean = self._vector(action_mean, 0.0, action_dim, "action_mean")
		self.action_std = self._vector(action_std, 1.0, action_dim, "action_std")
		self.min_std = self._vector(min_std, 0.05, action_dim, "min_std")
		self.action_low = self._optional_vector(action_low, action_dim, "action_low")
		self.action_high = self._optional_vector(action_high, action_dim, "action_high")
		if np.any(self.action_std <= 0) or np.any(self.min_std <= 0):
			raise ValueError("action_std and min_std must be positive")
		if (self.action_low is None) != (self.action_high is None):
			raise ValueError("action_low and action_high must be provided together")
		if self.action_low is not None:
			if np.any(self.action_low >= self.action_high):
				raise ValueError("Every action_low must be below action_high")
			self.action_mean = np.clip(self.action_mean, self.action_low, self.action_high)

		self.raw_action_mean = self._vector(
			raw_action_mean, 0.0, primitive_action_dim, "raw_action_mean"
		)
		self.raw_action_std = self._vector(
			raw_action_std, 1.0, primitive_action_dim, "raw_action_std"
		)
		if np.any(self.raw_action_std <= 0):
			raise ValueError("raw_action_std must be positive")
		if not 0.0 <= momentum < 1.0:
			raise ValueError(f"momentum must be in [0, 1), got {momentum}")
		self.momentum = float(momentum)
		if not 0.0 <= noise_correlation < 1.0:
			raise ValueError("noise_correlation must be in [0, 1)")
		if not 0.0 <= fresh_fraction < 1.0:
			raise ValueError("fresh_fraction must be in [0, 1)")
		self.noise_correlation = float(noise_correlation)
		self.fresh_fraction = float(fresh_fraction)
		self.separate_gripper_searches = bool(separate_gripper_searches)
		self.sampler = np.random.default_rng(seed)

	@staticmethod
	def _vector(value, default, size, name):
		array = np.asarray(default if value is None else value, dtype=np.float32)
		if array.ndim == 0:
			array = np.full(size, float(array), dtype=np.float32)
		if array.shape != (size,) or not np.isfinite(array).all():
			raise ValueError(f"{name} must be finite and have shape ({size},)")
		return array.copy()

	@classmethod
	def _optional_vector(cls, value, size, name):
		return None if value is None else cls._vector(value, 0.0, size, name)

	def normalize_primitive_actions(self, actions):
		actions = np.asarray(actions, dtype=np.float32)
		if actions.shape[-1] != self.primitive_action_dim:
			raise ValueError("primitive action dimension does not match planner")
		return (actions - self.raw_action_mean) / self.raw_action_std

	def denormalize_action_blocks(self, blocks):
		blocks = np.asarray(blocks, dtype=np.float32)
		if blocks.shape[-1] != self.action_dim:
			raise ValueError("flattened action block dimension does not match planner")
		primitive = blocks.reshape(*blocks.shape[:-1], self.block_size, self.primitive_action_dim)
		return primitive * self.raw_action_std + self.raw_action_mean

	def single_switch_gripper_schedules(self):
		"""Return coherent gripper schedules with at most one block-boundary switch."""
		total_steps = self.H * self.block_size
		schedules = []
		for start in (-1.0, 1.0):
			schedules.append(np.full(total_steps, start, dtype=np.float32))
			for switch in range(self.block_size, total_steps, self.block_size):
				schedule = np.full(total_steps, start, dtype=np.float32)
				schedule[switch:] = -start
				schedules.append(schedule)
		raw = np.stack(schedules).reshape(-1, self.H, self.block_size)
		return (raw - self.raw_action_mean[-1]) / self.raw_action_std[-1]

	def shifted_plan(self, plan):
		"""Discard the executed block and repeat the final block as the new tail."""
		plan = np.asarray(plan, dtype=np.float32)
		if plan.shape != (self.H, self.action_dim) or not np.isfinite(plan).all():
			raise ValueError(f"plan must have shape {(self.H, self.action_dim)}")
		return np.concatenate((plan[1:], plan[-1:]), axis=0)

	def _sample_noise(self, count):
		"""Unit-variance AR(1) noise across primitive time, including block boundaries."""
		noise = self.sampler.standard_normal(
			(count, self.H * self.block_size, self.primitive_action_dim)
		).astype(np.float32)
		rho = self.noise_correlation
		if rho:
			innovation_scale = np.sqrt(1.0 - rho * rho)
			for step in range(1, noise.shape[1]):
				noise[:, step] = rho * noise[:, step - 1] + innovation_scale * noise[:, step]
		return noise.reshape(count, self.H, self.action_dim)

	def _bound_actions(self, actions):
		if self.action_low is None:
			return actions
		low = torch.as_tensor(self.action_low, dtype=actions.dtype, device=actions.device)
		high = torch.as_tensor(self.action_high, dtype=actions.dtype, device=actions.device)
		return torch.maximum(torch.minimum(actions, high), low)

	def _expand_gripper_schedules(self, schedules, device):
		schedules = torch.as_tensor(schedules, dtype=torch.float32, device=device)
		if schedules.ndim == 2 and schedules.shape == (self.H, self.block_size):
			schedules = schedules.unsqueeze(0)
		if schedules.ndim != 3 or schedules.shape[0] < 1 or schedules.shape[1:] != (self.H, self.block_size):
			raise ValueError(
				"gripper_schedules must have shape "
				f"(S, {self.H}, {self.block_size}), got {tuple(schedules.shape)}"
			)
		if not torch.isfinite(schedules).all():
			raise ValueError("gripper_schedules must be finite")
		return schedules

	def _prepare_inputs(self, zi, a, zg):
		zi = torch.as_tensor(zi, dtype=torch.float32, device=self.device)
		a = torch.as_tensor(a, dtype=torch.float32, device=self.device)
		zg = torch.as_tensor(zg, dtype=torch.float32, device=self.device)
		if zi.ndim != 2:
			raise ValueError(f"zi must have shape (context, D), got {tuple(zi.shape)}")
		if a.ndim != 2 or a.shape != (zi.shape[0] - 1, self.action_dim):
			raise ValueError(
				f"a must have shape ({zi.shape[0] - 1}, {self.action_dim}), got {tuple(a.shape)}"
			)
		if zg.ndim == 1:
			zg = zg.unsqueeze(0)
		if zg.ndim != 2 or zg.shape[-1] != zi.shape[-1]:
			raise ValueError("zg must contain one or more embeddings matching zi")
		return zi, a, zg

	@torch.no_grad()
	def score_actions(self, zi, a, actions, zg, apply_bounds=True):
		"""Return terminal latent MSE for each standardized action sequence."""
		zi, a, zg = self._prepare_inputs(zi, a, zg)
		actions = torch.as_tensor(actions, dtype=torch.float32, device=self.device)
		if actions.ndim == 2:
			actions = actions.unsqueeze(0)
		if actions.ndim != 3 or actions.shape[2] != self.action_dim:
			raise ValueError("actions must have shape (H, A) or (N, H, A)")
		if apply_bounds:
			actions = self._bound_actions(actions)

		num_sequences = actions.shape[0]
		window = zi.unsqueeze(0).expand(num_sequences, -1, -1)
		action_history = a.unsqueeze(0).expand(num_sequences, -1, -1)
		max_context = int(getattr(self.pred, "window", zi.shape[0]))
		for step in range(actions.shape[1]):
			model_actions = torch.cat((action_history, actions[:, step:step + 1]), dim=1)
			prediction = self.pred(window, model_actions)[:, -1]
			window = torch.cat((window, prediction.unsqueeze(1)), dim=1)
			action_history = model_actions
			if window.shape[1] > max_context:
				window = window[:, -max_context:]
				action_history = action_history[:, -(max_context - 1):]

		goal_costs = (window[:, -1, None] - zg[None]).square().mean(dim=-1)
		return goal_costs.min(dim=1).values

	@torch.no_grad()
	def CEM(self, zi, a, zg, initial_mean=None, gripper_schedules=None):
		"""Search N total candidates; optionally fit separate motion distributions per schedule."""
		zi, a, zg = self._prepare_inputs(zi, a, zg)
		self.pred.eval()
		self.enc.eval()
		prior_mean = np.broadcast_to(self.action_mean, (self.H, self.action_dim)).copy()
		base_mean = prior_mean.copy()
		if initial_mean is not None:
			base_mean = np.asarray(initial_mean, dtype=np.float32).copy()
			if base_mean.shape != (self.H, self.action_dim) or not np.isfinite(base_mean).all():
				raise ValueError(f"initial_mean must have shape {(self.H, self.action_dim)}")
		if self.action_low is not None:
			base_mean = np.clip(base_mean, self.action_low, self.action_high)
		base_std = np.broadcast_to(self.action_std, base_mean.shape).copy()

		schedules = (
			None if gripper_schedules is None
			else self._expand_gripper_schedules(gripper_schedules, self.device)
		)
		separate = schedules is not None and self.separate_gripper_searches
		groups = schedules.shape[0] if separate else 1
		if separate and self.N < 2 * groups:
			raise ValueError(f"Need at least {2 * groups} total samples for {groups} gripper schedules")
		# Preserve the total population and elite fraction, including uneven splits.
		counts = np.full(groups, self.N // groups, dtype=int)
		counts[:self.N % groups] += 1
		offsets = np.concatenate(([0], np.cumsum(counts)))
		group_indices = np.repeat(np.arange(groups), counts)
		elite_counts = [
			min(int(count), max(2, int(np.ceil(self.K * count / self.N))))
			if separate else self.K for count in counts
		]
		mean = np.broadcast_to(base_mean, (groups, self.H, self.action_dim)).copy()
		std = np.broadcast_to(base_std, mean.shape).copy()
		best_samples = []
		if separate:
			candidate_schedules = schedules[torch.as_tensor(group_indices, device=self.device)]
		for _ in range(self.T):
			noise = self._sample_noise(self.N)
			samples = mean[group_indices] + std[group_indices] * noise
			for group, count in enumerate(counts):
				start, stop = int(offsets[group]), int(offsets[group + 1])
				# Keep some broad proposals so a bad warm start cannot trap every sample.
				fresh = min(int(count) - 1, int(np.ceil(self.fresh_fraction * count)))
				if fresh:
					samples[stop - fresh:stop] = prior_mean + base_std * noise[stop - fresh:stop]
				# Score the current mean too, including the exact warm start on iteration 1.
				samples[start] = mean[group]
			actions = torch.as_tensor(samples, dtype=torch.float32, device=self.device)
			actions = self._bound_actions(actions)
			if schedules is not None:
				if separate:
					actions[..., self.gripper_indices] = candidate_schedules
				else:
					schedule_indices = torch.as_tensor(
						self.sampler.permutation(self.N) % schedules.shape[0], device=self.device,
					)
					actions[..., self.gripper_indices] = schedules[schedule_indices]
			costs = self.score_actions(zi, a, actions, zg, apply_bounds=False)
			best_samples.append(actions[costs.argmin()].unsqueeze(0).clone())
			elite_means, elite_stds = [], []
			for group, k in enumerate(elite_counts):
				start, stop = int(offsets[group]), int(offsets[group + 1])
				elite_indices = torch.topk(costs[start:stop], k=k, largest=False).indices + start
				elites = actions[elite_indices]
				elite_means.append(elites.mean(dim=0))
				elite_stds.append(elites.std(dim=0, unbiased=False))
			elite_mean = torch.stack(elite_means).cpu().numpy()
			elite_std = torch.stack(elite_stds).cpu().numpy()
			mean = self.momentum * mean + (1.0 - self.momentum) * elite_mean
			std = np.maximum(
				self.momentum * std + (1.0 - self.momentum) * elite_std,
				self.min_std,
			)

		candidate_means = torch.as_tensor(mean, dtype=torch.float32, device=self.device)
		candidate_means = self._bound_actions(candidate_means)
		if schedules is not None:
			if not separate:
				candidate_means = candidate_means.expand(schedules.shape[0], -1, -1).clone()
			candidate_means[..., self.gripper_indices] = schedules
		final_candidates = torch.cat([candidate_means, *best_samples], dim=0)
		costs = self.score_actions(zi, a, final_candidates, zg, apply_bounds=False)
		best = final_candidates[costs.argmin()]
		return best.cpu().numpy().astype(np.float32, copy=False)


"""Actor initializes one complete plan; ordinary CEM then searches action space."""
import numpy as np
import torch



class ActorSeededPlanner(Planner):
    def __init__(self, *args, actor, mode='actor_seeded', **kwargs):
        super().__init__(*args, **kwargs)
        if mode not in ('actor_seeded', 'direct', 'cem'):
            raise ValueError('mode must be actor_seeded, direct, or cem')
        if self.action_low is None:
            raise ValueError('Actor planning requires action bounds')
        self.actor, self.mode = actor, mode

    def executable(self, actions):
        mean = torch.as_tensor(self.raw_action_mean, device=actions.device, dtype=actions.dtype)
        std = torch.as_tensor(self.raw_action_std, device=actions.device, dtype=actions.dtype)
        raw = (actions * std + mean).clamp(-1, 1)
        raw = torch.cat((raw[..., :-1], torch.where(raw[..., -1:] >= 0, 1., -1.)), -1)
        return (raw - mean) / std

    def _bound_actions(self, actions):
        bounded = super()._bound_actions(actions)
        return self.executable(bounded.reshape(*bounded.shape[:-1], self.block_size,
                                               self.primitive_action_dim)).reshape_as(bounded)

    @torch.no_grad()
    def actor_plan(self, zi, history, goal):
        states, actions = zi[None], history[None]
        low = torch.as_tensor(self.action_low, device=zi.device).reshape(self.block_size, -1)
        high = torch.as_tensor(self.action_high, device=zi.device).reshape(self.block_size, -1)
        blocks = []
        for _ in range(self.H):
            if actions.shape[1]:
                previous = actions[:, -1].reshape(1, self.block_size, self.primitive_action_dim)
            else:
                zero = (-torch.as_tensor(self.raw_action_mean, device=zi.device) /
                        torch.as_tensor(self.raw_action_std, device=zi.device))
                previous = zero.expand(1, self.block_size, -1)
            prefix = zi.new_empty(1, 0, self.primitive_action_dim)
            for j in range(self.block_size):
                means, _ = self.actor(states[:, -1], goal.reshape(1, -1), previous, prefix)
                proposal = self.executable(means[:, -1].clamp(low[j], high[j]))
                prefix = torch.cat((prefix, proposal[:, None]), dim=1)
            block = prefix.flatten(1)
            blocks.append(block[0])
            actions = torch.cat((actions, block[:, None]), dim=1)
            predicted = self.pred(states, actions)[:, -1]
            states = torch.cat((states, predicted[:, None]), dim=1)[:, -self.pred.window:]
            actions = actions[:, -(self.pred.window - 1):]
        return torch.stack(blocks).cpu().numpy()

    @torch.no_grad()
    def CEM(self, zi, a, zg, initial_mean=None, gripper_schedules=None):
        if initial_mean is not None or gripper_schedules is not None:
            raise ValueError('This planner uses actor initialization, not warm starts or gripper schedules')
        self.actor.eval()
        self.pred.eval()
        self.enc.eval()
        zi, a, goals = self._prepare_inputs(zi, a, zg)
        if goals.shape[0] != 1:
            raise ValueError('Expected exactly one active subgoal')
        if self.mode == 'cem':
            return super().CEM(zi, a, goals)
        initial_plan = self.actor_plan(zi, a, goals[0])
        if not np.isfinite(initial_plan).all():
            raise FloatingPointError('Nonfinite actor initialization')
        if self.mode == 'direct':
            return initial_plan
        # No actor calls below: Gaussian means/stds are fitted to elite ACTIONS.
        return super().CEM(zi, a, goals, initial_mean=initial_plan)


In [ ]:
import mujoco
import robosuite as suite
import robosuite.macros as macros
from robosuite.controllers import load_controller_config
from robosuite.environments.manipulation.lift import Lift
from mimicgen.envs.robosuite.single_arm_env_mg import SingleArmEnv_MG

class Lift_D0(Lift, SingleArmEnv_MG):
    def edit_model_xml(self, xml_str):
        return SingleArmEnv_MG.edit_model_xml(self, xml_str)

    def reward(self, action=None):
        return Lift.reward(self, action=action)

CAM_ABOVE = 'agentview'
CAM_WRIST = 'robot0_eye_in_hand'
IMAGE_SIZE = 84
macros.IMAGE_CONVENTION = 'opencv'

def select_checkpoint(model_dir: Path) -> Path:
	best = model_dir / "best_model.zip"
	if best.exists():
		return best
	checkpoints = []
	for path in model_dir.glob("epoch_*_model.zip"):
		match = re.fullmatch(r"epoch_(\d+)_model\.zip", path.name)
		if match:
			checkpoints.append((int(match.group(1)), path))
	if not checkpoints:
		raise FileNotFoundError(f"No primary checkpoint found in {model_dir}")
	return max(checkpoints)[1]


def _load_torch(zf, member, tmp, weights_only):
	zf.extract(member, tmp)
	try:
		return torch.load(tmp / member, map_location="cpu", weights_only=weights_only)
	except TypeError:
		return torch.load(tmp / member, map_location="cpu")


def load_models(checkpoint: Path, device):
	with zipfile.ZipFile(checkpoint, "r") as zf, tempfile.TemporaryDirectory() as tmp_name:
		tmp = Path(tmp_name)
		if "state_stats.pth" not in zf.namelist():
			raise ValueError("Checkpoint has no state/action statistics; retrain it")
		stats = _load_torch(zf, "state_stats.pth", tmp, True)
		expected = {
			"checkpoint_format_version": CHECKPOINT_FORMAT_VERSION,
			"architecture_id": ARCHITECTURE_ID,
			"action_encoding_id": ACTION_ENCODING_ID,
			"image_normalization_id": IMAGE_NORMALIZATION_ID,
			"frame_skip": FRAME_SKIP,
			"primitive_action_dim": PRIMITIVE_ACTION_DIM,
			"predictor_action_dim": PREDICTOR_ACTION_DIM,
			"action_smoothing_dim": ACTION_SMOOTHING_DIM,
		}
		for key, value in expected.items():
			if stats.get(key) != value:
				raise ValueError(
					f"Incompatible checkpoint: {key}={stats.get(key)!r}, expected {value!r}. "
					"This evaluator requires a newly trained corrected checkpoint."
				)

		actor = GoalActor(**stats["actor_config"]).to(device)
		actor.load_state_dict(_load_torch(zf, "actor_weights.pth", tmp, True))
		actor.eval()
		encoder = RobotEncoder().to(device)
		predictor = PredictorNetwork(d_action=PREDICTOR_ACTION_DIM, window=WINDOW, action_smoothing_dim=ACTION_SMOOTHING_DIM).to(device)
		encoder.load_state_dict(_load_torch(zf, "encoder_weights.pth", tmp, True))
		predictor.load_state_dict(_load_torch(zf, "predictor_weights.pth", tmp, True))

	encoder.eval()
	predictor.eval()
	return encoder, predictor, actor, {key: torch.as_tensor(stats[key]).float() for key in (
		"state_mean", "state_std", "action_mean", "action_std", "action_low", "action_high"
	)}


def set_geom_groups(env, collision=False, visual=True):
	context = env.sim._render_context_offscreen
	if context is None:
		return
	address = context.vopt.geomgroup.__array_interface__["data"][0]
	groups = (ctypes.c_uint8 * 6).from_address(address)
	for index in range(6):
		groups[index] = 0
	groups[0] = int(collision)
	groups[1] = int(visual)


def observation_tensors(obs, device, stats):
	above = torch.from_numpy(obs[f"{CAM_ABOVE}_image"]).permute(2, 0, 1).float() / 255.0
	wrist = torch.from_numpy(obs[f"{CAM_WRIST}_image"]).permute(2, 0, 1).float() / 255.0
	above = normalize_robot_images(above)
	wrist = normalize_robot_images(wrist)
	state = build_robot_state(
		torch.from_numpy(obs["robot0_eef_pos"]).float(),
		torch.from_numpy(obs["robot0_eef_quat"]).float(),
		torch.from_numpy(obs["robot0_gripper_qpos"]).float(),
	)
	state = normalize_robot_state(state, stats["state_mean"], stats["state_std"])
	return above[None, None].to(device), wrist[None, None].to(device), state[None, None].to(device)


@torch.no_grad()
def encode_observation(encoder, obs, device, stats):
	return encoder(*observation_tensors(obs, device, stats))[0, 0]


@torch.no_grad()
def encode_demo_frames(encoder, obs, indices, device, stats, batch_size=128):
	outputs = []
	for offset in range(0, len(indices), batch_size):
		batch_indices = indices[offset:offset + batch_size]
		above = torch.from_numpy(obs["agentview_image"][batch_indices]).permute(0, 3, 1, 2).float() / 255.0
		wrist = torch.from_numpy(obs["robot0_eye_in_hand_image"][batch_indices]).permute(0, 3, 1, 2).float() / 255.0
		above = normalize_robot_images(above).unsqueeze(1).to(device)
		wrist = normalize_robot_images(wrist).unsqueeze(1).to(device)
		state = build_robot_state(
			torch.from_numpy(obs["robot0_eef_pos"][batch_indices]).float(),
			torch.from_numpy(obs["robot0_eef_quat"][batch_indices]).float(),
			torch.from_numpy(obs["robot0_gripper_qpos"][batch_indices]).float(),
		)
		state = normalize_robot_state(state, stats["state_mean"], stats["state_std"])
		outputs.append(encoder(above, wrist, state[:, None].to(device))[:, 0].cpu())
	return torch.cat(outputs).to(device)


def _text(value):
	return value.decode("utf-8") if isinstance(value, bytes) else str(value)


def load_subgoal_labels(path):
	path = Path(path)
	if not path.exists():
		raise FileNotFoundError(
			f"Manual subgoal labels not found: {path}. Run label_subgoals.py first."
		)
	with Path(path).open("r", encoding="utf-8") as stream:
		data = json.load(stream)
	if data.get("format_version") != 1 or not isinstance(data.get("labels"), dict):
		raise ValueError(f"Unsupported subgoal label file format: {path}")
	return data["labels"]


@torch.no_grad()
def load_eval_bank(encoder, path, labels_path, device, stats, demo_name=None):
	labels = load_subgoal_labels(labels_path)
	entries = []
	with h5py.File(path, "r") as file:
		if _text(file["data"].attrs.get("env", "")) != ENV_NAME:
			raise ValueError(f"{path} does not contain {ENV_NAME} evals")
		version = file["data"].attrs.get("mujoco_version")
		if version is not None and _text(version) != mujoco.__version__:
			print(f"WARNING: evals use MuJoCo {_text(version)}; inference uses {mujoco.__version__}")
		if demo_name is not None:
			if demo_name not in labels:
				raise KeyError(f"{demo_name!r} has no manual grasp label in {labels_path}")
			names = [demo_name]
		elif "mask/eval" in file:
			all_names = [_text(value) for value in file["mask/eval"][:]]
			names = [name for name in all_names if name in labels]
		else:
			all_names = sorted(file["data"].keys())
			names = [name for name in all_names if name in labels]
		if not names:
			raise ValueError(f"No labeled eval demos from {path} were found in {labels_path}")
		if demo_name is None and len(names) != len(all_names):
			print(f"using {len(names)}/{len(all_names)} eval demos with manual grasp labels")
		for name in names:
			demo = file[f"data/{name}"]
			num_frames = int(demo["states"].shape[0])
			label = labels[name]
			grasp_index = int(label["grasp_frame"])
			goal_index = int(label.get("goal_frame", num_frames - 1))
			if "num_frames" in label and int(label["num_frames"]) != num_frames:
				raise ValueError(
					f"{name}: labels describe {label['num_frames']} frames, eval has {num_frames}"
				)
			if not 0 < grasp_index < goal_index < num_frames:
				raise ValueError(
					f"{name}: expected 0 < grasp_frame < goal_frame < {num_frames}, "
					f"got {grasp_index} and {goal_index}"
				)
			if goal_index != num_frames - 1:
				raise ValueError(
					f"{name}: goal_frame must be the final frame ({num_frames - 1}), got {goal_index}"
				)
			embeddings = encode_demo_frames(
				encoder, demo["obs"], [0, grasp_index, goal_index], device, stats
			)
			model_xml = _text(demo.attrs["model_file"])
			entries.append({
				"name": name,
				"start": embeddings[0],
				"stages": (
					{"name": "grasp", "goal": embeddings[1], "frame": grasp_index},
					{"name": "lift", "goal": embeddings[2], "frame": goal_index},
				),
				"initial_state": np.asarray(demo["states"][0]).copy(),
				"model_xml": model_xml,
			})
	return entries


def make_env(horizon):
	controller = load_controller_config(default_controller="OSC_POSE")
	env = suite.make(
		ENV_NAME, robots="Panda", controller_configs=controller,
		has_renderer=False, has_offscreen_renderer=True,
		render_collision_mesh=False, render_visual_mesh=True,
		use_camera_obs=True, camera_names=[CAM_ABOVE, CAM_WRIST],
		camera_heights=IMAGE_SIZE, camera_widths=IMAGE_SIZE,
		control_freq=CONTROL_FREQ, horizon=horizon, ignore_done=True,
	)
	set_geom_groups(env)
	return env


def reset_to_eval_state(env, entry):
	env.reset()
	env.reset_from_xml_string(env.edit_model_xml(entry["model_xml"]))
	env.sim.reset()
	env.sim.set_state_from_flattened(entry["initial_state"])
	env.sim.forward()
	set_geom_groups(env)
	return env._get_observations(force_update=True)


def render_frame(env, size=256):
	views = [
		env.sim.render(height=size, width=size, camera_name=camera)[::-1]
		for camera in (CAM_ABOVE, CAM_WRIST)
	]
	return np.concatenate(views, axis=1).astype(np.uint8)


def postprocess_action(action):
	action = np.clip(np.asarray(action, dtype=np.float64), -1.0, 1.0).copy()
	action[-1] = 1.0 if action[-1] >= 0 else -1.0
	return action


def held_gripper_schedule(planner, command):
	"""Keep a specified gripper command fixed throughout a CEM plan."""
	raw = np.full((1, planner.H, planner.block_size), command, dtype=np.float32)
	return (raw - planner.raw_action_mean[-1]) / planner.raw_action_std[-1]


def run_rollout(
	env,
	planner,
	encoder,
	entry,
	device,
	stats,
	writer,
	stage_settings,
	warm_start=True,
):
	for stage in entry["stages"]:
		if stage["name"] not in stage_settings:
			raise KeyError(f"Missing settings for {stage['name']!r} stage")
		settings = stage_settings[stage["name"]]
		threshold = float(settings["threshold"])
		if int(settings["budget"]) < 1 or not np.isfinite(threshold) or threshold < 0:
			raise ValueError(f"Invalid settings for {stage['name']!r} stage")
		if float(settings["budget"]) % FRAME_SKIP:
			raise ValueError(f"Stage budgets must be multiples of {FRAME_SKIP} primitive steps")
	obs = reset_to_eval_state(env, entry)
	writer.append_data(render_frame(env))
	z = encode_observation(encoder, obs, device, stats)
	print(f"exact eval {entry['name']} | start embedding MSE: {(z-entry['start']).square().mean().item():.6f}")
	context = deque([z], maxlen=WINDOW)
	action_history = deque(maxlen=WINDOW - 1)
	total_steps = 0

	for number, stage in enumerate(entry["stages"], 1):
		# Warm starts belong to this rollout and goal; never carry grasp plans into lift.
		previous_plan = None
		goal = stage["goal"]
		budget = int(stage_settings[stage["name"]]["budget"])
		threshold = float(stage_settings[stage["name"]]["threshold"])
		use_threshold = bool(stage_settings[stage["name"]].get("use_threshold", True))
		steps = 0
		current_mse = (context[-1] - goal).square().mean().item()
		reached = use_threshold and current_mse <= threshold
		threshold_text = f"{threshold:.4f}" if use_threshold else "disabled"
		print(
			f"{stage['name']} stage {number}/{len(entry['stages'])}: "
			f"demo frame {stage['frame']}, threshold={threshold_text}"
		)
		if reached:
			print(f"  already within threshold (latent MSE {current_mse:.4f})")
		while steps < budget and not reached:
			zi = torch.stack(tuple(context))
			a = (
				torch.stack(tuple(action_history))
				if action_history else torch.empty(0, PREDICTOR_ACTION_DIM, device=device)
			)
			plan = planner.CEM(zi, a, goal)
			previous_plan = plan
			raw_blocks = planner.denormalize_action_blocks(plan)
			# Execute one ordered 5-action block, then replan from observation.
			for raw_block in raw_blocks[:1]:
				executed = []
				for raw_action in raw_block:
					action = postprocess_action(raw_action)
					obs, _, done, _ = env.step(action)
					set_geom_groups(env)
					writer.append_data(render_frame(env))
					executed.append(action)
					steps += 1
					total_steps += 1
					if env._check_success():
						return True, total_steps
					if done:
						return False, total_steps
					if done or steps >= budget:
						break

				normalized_block = planner.normalize_primitive_actions(np.stack(executed))
				if len(executed) == FRAME_SKIP:
					action_history.append(torch.as_tensor(
						normalized_block.reshape(-1), dtype=z.dtype, device=device
					))
				z = encode_observation(encoder, obs, device, stats)
				context.append(z)
				goal_mse = (z - goal).square().mean().item()
				print(f"  steps {steps:>3}/{budget} | latent MSE {goal_mse:.4f}")
				if use_threshold and goal_mse <= threshold:
					reached = True
					break
				if done or steps >= budget:
					break
		if not use_threshold:
			print(f"  fixed {stage['name']} budget complete ({steps}/{budget} steps)")
			continue
		if not reached:
			print("subgoal budget exhausted without reaching the latent goal")
			return False, total_steps
	return bool(env._check_success()), total_steps


def next_rollout_number(rollout_dir: Path) -> int:
	"""Continue after the largest completed rollout_N_pass/fail video."""
	latest = 0
	# Search anywhere in the name so videos from the earlier
	# <checkpoint>_<demo>_rollout_N_<result> convention are counted too.
	pattern = re.compile(r"(?:^|_)rollout_(\d+)_(?:pass|fail)\.mp4$")
	for path in rollout_dir.glob("*rollout_*_*.mp4"):
		match = pattern.search(path.name)
		if match:
			latest = max(latest, int(match.group(1)))
	return latest + 1


In [ ]:
EVAL_HDF5 = PRIMARY_DIR / 'evals.hdf5'
SUBGOAL_LABELS = PRIMARY_DIR / 'subgoal_labels.json'
if not EVAL_HDF5.exists():
    raise FileNotFoundError(f'Put evals.hdf5 in {PRIMARY_DIR}')
if not SUBGOAL_LABELS.exists():
    raise FileNotFoundError(
        f'Put subgoal_labels.json from label_subgoals.py in {PRIMARY_DIR}'
    )
print(f'evals: {EVAL_HDF5}')
print(f'labels: {SUBGOAL_LABELS}')


## Rollouts

PLANNER_MODE="actor_seeded" initializes ordinary action-space CEM using one actor plan.
"direct" executes actor-only plans; "cem" uses ordinary CEM without actor initialization.
Defaults: 300 candidates, 30 elites, 10 iterations. Gaussian initial standard deviation is
the ordinary planner default (1.0 in standardized action coordinates), not the actor's
predicted uncertainty. No residual-temperature scaling. The original actor proposal is
scored in iteration one, and best candidates are retained.
Checkpoint selection, manual grasp/lift goals, thresholds OFF, and 150/100 allowances
remain unchanged. Videos continue the existing rollout_N_pass/fail.mp4 numbering.


In [ ]:
# ----------------------- rollout controls -----------------------
# CHECKPOINT_EPOCH refers to actor-training epochs, starting at 0.
# Set to None to load best_model.zip instead of an explicit epoch checkpoint.
CHECKPOINT_EPOCH = None

NUM_ROLLOUTS = 50
EVAL_DEMO = None               # None visits labeled demos in numeric order; a name repeats that demo.
EVAL_SEED = 42                 # CEM sampling seed; demo order is deterministic.

# False: use fixed stage budgets; MSE is printed but never causes a switch or stop.
USE_LATENT_THRESHOLDS = False

GRASP_BUDGET = 150             # Primitive steps allowed to reach the labeled grasp.
GRASP_THRESHOLD = 0.05         # Advance when latent MSE is at or below this value.
LIFT_BUDGET = 100              # Primitive steps allowed to reach the final lifted frame.
LIFT_THRESHOLD = 0.05

HORIZON = 5                   # Lookahead in 5-action blocks: 2=10 steps, 3=15, 5=25.
PLANNER_MODE = "actor_seeded"  # "actor_seeded", "direct", or "cem"
NUM_SAMPLES = 300
NUM_ELITES = 30
CEM_ITERS = 10

# ---------------------------------------------------------------

if DEVICE != 'cuda':
    raise RuntimeError('Select a GPU runtime for rollouts')

if not isinstance(HORIZON, int) or isinstance(HORIZON, bool) or HORIZON < 1:
    raise ValueError('HORIZON must be a positive integer number of 5-action blocks')
PLAN_STEPS = HORIZON * FRAME_SKIP
print(f'lookahead: {PLAN_STEPS} primitive steps; execute {FRAME_SKIP} before replanning')

if CHECKPOINT_EPOCH is None:
    CHECKPOINT = select_checkpoint(MODEL_DIR)
else:
    if not isinstance(CHECKPOINT_EPOCH, int) or isinstance(CHECKPOINT_EPOCH, bool) or CHECKPOINT_EPOCH < 0:
        raise ValueError('CHECKPOINT_EPOCH must be a nonnegative integer or None')
    CHECKPOINT = MODEL_DIR / f'epoch_{CHECKPOINT_EPOCH}_model.zip'
    if not CHECKPOINT.exists():
        raise FileNotFoundError(
            f'Requested epoch {CHECKPOINT_EPOCH}, but checkpoint does not exist: {CHECKPOINT}'
        )

# Reload every model-dependent object in this cell. This is important because
# goal embeddings from one encoder must never be reused with another epoch.
encoder, predictor, actor, STATS = load_models(CHECKPOINT, DEVICE)
print(f'checkpoint: {CHECKPOINT}')

for stage_name, budget, threshold in (
    ('grasp', GRASP_BUDGET, GRASP_THRESHOLD),
    ('lift', LIFT_BUDGET, LIFT_THRESHOLD),
):
    if budget < 1:
        raise ValueError(f'{stage_name.upper()}_BUDGET must be positive')
    if threshold < 0:
        raise ValueError(f'{stage_name.upper()}_THRESHOLD must be nonnegative')
    if budget % FRAME_SKIP:
        raise ValueError(f'{stage_name.upper()}_BUDGET must be a multiple of {FRAME_SKIP}')

primitive_mean = STATS['action_mean'].numpy()
primitive_std = STATS['action_std'].numpy()
normalized_low = ((STATS['action_low'] - STATS['action_mean']) / STATS['action_std']).numpy()
normalized_high = ((STATS['action_high'] - STATS['action_mean']) / STATS['action_std']).numpy()
planner = ActorSeededPlanner(
    DEVICE, encoder, predictor, actor=actor, mode=PLANNER_MODE,
    H=HORIZON, N=NUM_SAMPLES, K=NUM_ELITES, T=CEM_ITERS,
    action_dim=PREDICTOR_ACTION_DIM,
    primitive_action_dim=PRIMITIVE_ACTION_DIM,
    action_low=np.tile(normalized_low, FRAME_SKIP),
    action_high=np.tile(normalized_high, FRAME_SKIP),
    raw_action_mean=primitive_mean,
    raw_action_std=primitive_std,
    min_std=0.05,
    momentum=0.1,
    seed=EVAL_SEED,
    noise_correlation=0.0, fresh_fraction=0.0, separate_gripper_searches=False,
)
entries = load_eval_bank(
    encoder, EVAL_HDF5, SUBGOAL_LABELS, DEVICE, STATS, EVAL_DEMO
)
stage_settings = {
    'grasp': {
        'budget': GRASP_BUDGET,
        'threshold': GRASP_THRESHOLD,
        'use_threshold': USE_LATENT_THRESHOLDS,
    },
    'lift': {
        'budget': LIFT_BUDGET,
        'threshold': LIFT_THRESHOLD,
        'use_threshold': USE_LATENT_THRESHOLDS,
    },
}
entries.sort(key=lambda entry: int(re.fullmatch(r'demo_(\d+)', entry['name']).group(1)))
if not isinstance(NUM_ROLLOUTS, int) or isinstance(NUM_ROLLOUTS, bool) or NUM_ROLLOUTS < 1:
    raise ValueError('NUM_ROLLOUTS must be a positive integer')
if EVAL_DEMO is None:
    if NUM_ROLLOUTS > len(entries):
        raise ValueError(f'Requested {NUM_ROLLOUTS} unique evals, but only {len(entries)} labeled demos are available')
    indices = range(NUM_ROLLOUTS)
else:
    indices = [0] * NUM_ROLLOUTS
print(f'Evaluating {NUM_ROLLOUTS} rollouts in demo order: {entries[indices[0]]["name"]} through {entries[indices[-1]]["name"]}')
env = make_env(max(200, GRASP_BUDGET + LIFT_BUDGET))

first_rollout_number = next_rollout_number(ROLLOUT_DIR)
print(
    f'completed rollout videos found through #{first_rollout_number - 1}; '
    f'starting at rollout_{first_rollout_number}'
)
successes = 0
started = time.perf_counter()
try:
    for batch_index, index in enumerate(indices):
        rollout_number = first_rollout_number + batch_index
        entry = entries[int(index)]
        pending = ROLLOUT_DIR / f'rollout_{rollout_number}_pending.mp4'
        pending.unlink(missing_ok=True)
        writer = imageio.get_writer(str(pending), fps=CONTROL_FREQ, codec='libx264')
        try:
            success, steps = run_rollout(
                env, planner, encoder, entry, DEVICE, STATS, writer, stage_settings,
                warm_start=False,
            )
        finally:
            writer.close()
        result = ROLLOUT_DIR / (
            f"rollout_{rollout_number}_{'pass' if success else 'fail'}.mp4"
        )
        result.unlink(missing_ok=True)
        pending.replace(result)
        successes += int(success)
        print(
            f"batch {batch_index + 1}/{NUM_ROLLOUTS} → rollout_{rollout_number}: "
            f"{'pass' if success else 'fail'} ({steps} steps, eval {entry['name']})"
        )
        print(f'video: {result}')
finally:
    env.close()
print(f'success rate: {successes}/{NUM_ROLLOUTS} = {100*successes/NUM_ROLLOUTS:.1f}%')
print(f'total inference time: {time.perf_counter()-started:.2f}s')


## Benchmark one planning call
Run the rollout cell first (set NUM_ROLLOUTS=1 for a quick trial). Times the selected planner mode with GPU synchronization.


In [ ]:
def synchronize():
    if DEVICE.startswith('cuda'):
        torch.cuda.synchronize()

benchmark_zi = torch.randn(1, EMBEDDING_DIM, device=DEVICE)
benchmark_actions = torch.empty(0, PREDICTOR_ACTION_DIM, device=DEVICE)
benchmark_goal = torch.randn(EMBEDDING_DIM, device=DEVICE)
print('warming up planner...')
_ = planner.CEM(benchmark_zi, benchmark_actions, benchmark_goal)
synchronize()
start = time.perf_counter()
planned_actions = planner.CEM(
    benchmark_zi, benchmark_actions, benchmark_goal
)
synchronize()
elapsed = time.perf_counter() - start
print(f'one planning call: {elapsed:.3f}s')
print(f'standardized predictor plan: {planned_actions.shape}')
print(f'decoded execution plan: {planner.denormalize_action_blocks(planned_actions).shape}')
print(f'lookahead: {planner.H * planner.block_size} primitive steps; execute {planner.block_size} before replanning')
